# Convert NAT-Files (HRV Band) to GeoTiff as multi-threaded Process
## Warp and Clip to Region Western Austria
Conda-Env: Scikit-Learn
Requirements: Windows 7-zip Installation (64bit)

In [1]:
# import necessary python packages
### import py7zr
import os
import gzip
import subprocess
import shutil
import numpy as np
import gdal
import osr
import pyresample as pr
from satpy import Scene
import datetime
from datetime import timedelta
from multiprocessing.dummy import Pool as ThreadPool
import time
import pandas as pd

In [2]:
# template from https://www.dariusgoergen.com/contents/blog/2020-06-14-nat2tif/
def wrapper_nat2geotiff(eumetsat_native_output_path, eumetsat_geotiff_timestamped_path, eumetsat_archive_path, filename):
    print("# Process File:  " + filename)

    # check if File is a GZ or 7Z-File
    if (filename.endswith(".gz")) or filename.endswith(".7z") or filename.endswith(".zip") or filename.endswith(".bz2"):
        # get plain Filename without extension
        if (filename.endswith(".gz")) or filename.endswith(".7z"):
            plain_filename= os.path.basename(filename)[:-7]
        else:
            plain_filename= os.path.basename(filename)[:-8]

        # Create rounded Timestamp (to Quarter before) as result TIF-filename
        rounded_timestamp_filename = round_filename_to_quarter(filename = os.path.basename(filename)[0:74])

        # declare area variables
        areas = ["Vorarlberg","Lake of Constance"]

        # loop trought areas for clipped geotiff export
        for area in areas:        
            # create some information on the reference system
            area_id = "Austria West"
            description = "Geographical Coordinate System clipped on Western Austria Region"
            proj_id = "Austria"
            # specifing some parameters of the projection
            proj_dict = {"proj": "longlat", "ellps": "WGS84", "datum": "WGS84"}

            if area == "Vorarlberg":
                # calculate the width and height of the aoi in pixels - region Vorarlberg
                llx = 9 # lower left x coordinate in degrees
                lly = 46 # lower left y coordinate in degrees
                urx = 11 # upper right x coordinate in degrees
                ury = 48 # upper right y coordinate in degrees
                extend='Clip_Vorarlberg'
            else:
                # calculate the width and height of the aoi in pixels - lake of constance
                llx = 7 # lower left x coordinate in degrees
                lly = 45 # lower left y coordinate in degrees
                urx = 13 # upper right x coordinate in degrees
                ury = 50 # upper right y coordinate in degrees
                extend='Clip_Lake_of_Constance'

            # resolution = 0.005 # target resolution in degrees
            resolution = 0.01 # target resolution in degrees
            # calculating the number of pixels
            width = int((urx - llx) / resolution)
            height = int((ury - lly) / resolution)
            area_extent = (llx,lly,urx,ury)
            # defining the area
            area_def = pr.geometry.AreaDefinition(area_id, proj_id, description, proj_dict, width, height, area_extent)
            print(area_def)

            datasets = ['IR_VIS_WR','HRV']
            #datasets = ['IR_VIS_WR']

            for dataset in datasets:
                # set filenames as variables 
                archive_filename = os.path.join(eumetsat_archive_path, filename)
                geotiff_filename = os.path.join(eumetsat_geotiff_timestamped_path, extend, rounded_timestamp_filename + "_{}.tif".format(dataset))
                native_filename = os.path.join(eumetsat_native_output_path, plain_filename + ".nat")
                
                # GeoTIFF-File exists?
                if not os.path.isfile(geotiff_filename):
                    print("    GeoTIFF-File {} not exists!".format(geotiff_filename))
                    # NAT-File exists?
                    if not os.path.isfile(native_filename):
                            print("    Extract Native-File {} !".format(native_filename))
                            # use of 7z.exe for performance purpose
                            unzip_command = ['C:\\Program Files\\7-Zip\\7z.exe', 'e', "-o"+ eumetsat_native_output_path, archive_filename, '-y']
                            subprocess.call(unzip_command)

                    # Convert NAT-File to GeoTIFF-Format
                    reader = "seviri_l1b_native"
                    print("    Convert NAT-File to GeoTIFF: {}".format(geotiff_filename))
                    nat2tif(file = native_filename, calibration = "radiance", area_def = area_def, dataset = dataset, \
                            reader = reader, outdir = eumetsat_geotiff_timestamped_path, label = dataset, \
                        dtype = "float32", radius = 16000, epsilon = 0.5, nodata = -3.4E+38, outfile = geotiff_filename)
        
        # delete temporary compressed NAT-File
        print("### NAT-File: {}".format(native_filename))
        if os.path.exists(native_filename):
            try:
                print("    Deleting NAT-File: {}".format(native_filename))
                os.remove(native_filename)
            except:
                print("    Error: NAT-File {} is corrupted or locked! Skipping...".format(native_filename))
                shutil.copyfile(native_filename, os.path.join(eumetsat_archive_path + "\\..\\defekt", plain_filename + ".nat"))
                os.remove(native_filename)
                pass

In [3]:
def nat2tif(file, calibration, area_def, dataset, reader, outdir, label, dtype, radius, epsilon, nodata, outfile):
  # open the file
  scn = Scene(filenames = {reader: [file]})
 
  # set bands for each dataset
  if dataset == 'HRV':
    bands = ['HRV']
  else:
    # wrong order of bands - alphabetical
    #bands = ['IR_016','IR_039','IR_087','IR_097','IR_108','IR_120','IR_134','VIS006','VIS008','WV_062','WV_073']
    # correct order of bands
    # source: https://eumetsat.int/0-degree-service
    bands = ['VIS006','VIS008','IR_016','IR_039','WV_062','WV_073','IR_087','IR_097','IR_108','IR_120','IR_134']
    
  # set starting band for iteration
  bandnr = 1

  for band in bands:
     # let us check that the specified data set is actually available
    scn_names = scn.all_dataset_names()
    # raise exception if dataset is not present in available names
    if band not in scn_names:
      raise Exception("Specified dataset is not available.")
    
    # output band name
    print("       Execute Band {} as Bandnr.{}".format(band,bandnr))
    # we need to load the data, different calibration can be chosen
    scn.load([band], calibration=calibration)
    # let us extract the longitude and latitude data
    lons, lats = scn[band].area.get_lonlats()
    # now we can apply a swath definition for our output raster
    swath_def = pr.geometry.SwathDefinition(lons=lons, lats=lats)
    # and finally we also extract the data
    values = scn[band].values
    # we will now change the datatype of the arrays
    # depending on the present data this can be changed
    lons = lons.astype(dtype)
    lats = lats.astype(dtype)
    values = values.astype(dtype)

    # now we can already resample our data to the area of interest
    values = pr.kd_tree.resample_nearest(swath_def, values,
                                              area_def,
                                              radius_of_influence=radius, # in meters
                                              epsilon=epsilon,
                                              fill_value=False)
    # we are going to check if the outdir exists and create it if it doesnt
    
    print("       Band values:",band, np.min(values),np.max(values))
    
    if not os.path.exists(outdir):
      os.makedir(outdir)
    
    # now we define some metadata for our raster file
    cols = values.shape[1]
    rows = values.shape[0]
    pixelWidth = (area_def.area_extent[2] - area_def.area_extent[0]) / cols
    pixelHeight = (area_def.area_extent[1] - area_def.area_extent[3]) / rows
    originX = area_def.area_extent[0]
    originY = area_def.area_extent[3] 
    # create output is just for the first band necessary
    if bandnr == 1:
        # you can change the dataformat but be sure to be able to store negative values including -9999
        dst_datatype = gdal.GDT_Float32
        # here we actually create the file
        driver = gdal.GetDriverByName("GTiff")
        # GeoTIFF Options from https://kokoalberti.com/articles/geotiff-compression-optimization-guide/
        outRaster = driver.Create(outfile, cols, rows, len(bands), dst_datatype, [ 'COMPRESS=ZSTD', 'PREDICTOR=3', 'TILED=YES', 'NUM_THREADS=ALL_CPUS' ] )
        # writing the metadata
        outRaster.SetGeoTransform((originX, pixelWidth, 0, originY, 0, pixelHeight))
    # set band name to geotiff band description
    outRaster.GetRasterBand(bandnr).SetDescription(band)
    # creating a new band and writting the data
    outband = outRaster.GetRasterBand(bandnr)
    outband.WriteArray(np.array(values)) # writting the values
    outband.SetNoDataValue(nodata) #specified no data value by user
    outRasterSRS = osr.SpatialReference() # create CRS instance
    outRasterSRS.ImportFromEPSG(4326) # get info for EPSG 4326
    outRaster.SetProjection(outRasterSRS.ExportToWkt()) # set CRS as WKT
    # increase the bandnr
    bandnr = bandnr + 1
  # clean up
  outband = None
  outRaster.FlushCache()
  outRaster = None
  del file,  scn, outband, outRaster

In [4]:
def round_filename_to_quarter(filename):
    date_object = datetime.datetime.strptime(filename[24:38], '%Y%m%d%H%M%S')
    rounded = date_object - (date_object - date_object.min) % timedelta(minutes=15)
    print("    File-Timestamp:    " + date_object.strftime("%Y-%m-%d %H_%M_%S"))  # printed in default formatting
    rounded_str=rounded.strftime("%Y-%m-%d %H_%M_%S")
    print("    Rounded-Timestamp: " + rounded_str)  # printed in default formatting
    return rounded_str

In [6]:
if __name__ == '__main__':
    from concurrent.futures import ThreadPoolExecutor
    from concurrent.futures import as_completed
    from itertools import repeat
    
    # set variables
    eumetsat_path = "C:\\Users\\Andreas\\Documents\\UNIGIS\\2017\\Master-Thesis\\Daten\\Satellite\\EUMETSAT"
    eumetsat_native_output_path = eumetsat_path + "\\Native"
    eumetsat_geotiff_timestamped_path = eumetsat_path + "\\Result_Timestamped\\GeoTIFF\\TEST"
    eumetsat_archive_path = "F:\\EUMETSAT\\TEST"
    
    # create output folder
    if not os.path.exists(eumetsat_native_output_path):
         os.makedirs(eumetsat_native_output_path)
    if not os.path.exists(eumetsat_geotiff_timestamped_path):
         os.makedirs(eumetsat_geotiff_timestamped_path)
    
    # create empty list for files
    files = []
        
    # Loop all compressed NAT-Files
    for file in os.listdir(eumetsat_archive_path):
        files.append(file)
    
    # create multiple Threads for parallel processing
    with ThreadPoolExecutor(max_workers = 8) as executor:
        results = executor.map(wrapper_nat2geotiff, repeat(eumetsat_native_output_path), repeat(eumetsat_geotiff_timestamped_path), repeat(eumetsat_archive_path), files)
    for result in results:
        print(result)

# Process File:  MSG4-SEVI-MSG15-0100-NA-20181231121243.496000000Z-20181231121300-1404462-6.nat.gz# Process File:  MSG4-SEVI-MSG15-0100-NA-20181231122743.241000000Z-20181231122800-1404462-6.nat.gz
    File-Timestamp:    2018-12-31 12_27_43# Process File:  MSG4-SEVI-MSG15-0100-NA-20181231124242.986000000Z-20181231124300-1404462-6.nat.gz


    File-Timestamp:    2018-12-31 12_42_42
    Rounded-Timestamp: 2018-12-31 12_30_00
    File-Timestamp:    2018-12-31 12_12_43# Process File:  MSG4-SEVI-MSG15-0100-NA-20181231125742.731000000Z-20181231125759-1404462-6.nat.gz    Rounded-Timestamp: 2018-12-31 12_15_00

    File-Timestamp:    2018-12-31 12_57_42
    Rounded-Timestamp: 2018-12-31 12_45_00
# Process File:  MSG4-SEVI-MSG15-0100-NA-20181231131242.476000000Z-20181231131300-1404462-6.nat.gz
    File-Timestamp:    2018-12-31 13_12_42
    Rounded-Timestamp: 2018-12-31 13_00_00

    Rounded-Timestamp: 2018-12-31 12_00_00# Process File:  MSG4-SEVI-MSG15-0100-NA-20181231132742.221000000Z-201812311

C:\Users\Andreas\anaconda3\envs\scikit-learn\lib\site-packages\pyproj\crs\crs.py:543: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj_string = self.to_proj4()
C:\Users\Andreas\anaconda3\envs\scikit-learn\lib\site-packages\pyproj\crs\crs.py:543: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj_string = self.to_proj4()
C:\Users\Andreas\anaconda3\envs\scikit-learn\lib\site-packages\pyproj\crs\crs.py:543: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj_string = se

    Convert NAT-File to GeoTIFF: C:\Users\Andreas\Documents\UNIGIS\2017\Master-Thesis\Daten\Satellite\EUMETSAT\Result_Timestamped\GeoTIFF\TEST\Clip_Vorarlberg\2018-12-31 12_15_00_IR_VIS_WR.tif
    Convert NAT-File to GeoTIFF: C:\Users\Andreas\Documents\UNIGIS\2017\Master-Thesis\Daten\Satellite\EUMETSAT\Result_Timestamped\GeoTIFF\TEST\Clip_Vorarlberg\2018-12-31 12_30_00_IR_VIS_WR.tif    Convert NAT-File to GeoTIFF: C:\Users\Andreas\Documents\UNIGIS\2017\Master-Thesis\Daten\Satellite\EUMETSAT\Result_Timestamped\GeoTIFF\TEST\Clip_Vorarlberg\2018-12-31 12_45_00_IR_VIS_WR.tif    Convert NAT-File to GeoTIFF: C:\Users\Andreas\Documents\UNIGIS\2017\Master-Thesis\Daten\Satellite\EUMETSAT\Result_Timestamped\GeoTIFF\TEST\Clip_Vorarlberg\2018-12-31 13_00_00_IR_VIS_WR.tif


    Convert NAT-File to GeoTIFF: C:\Users\Andreas\Documents\UNIGIS\2017\Master-Thesis\Daten\Satellite\EUMETSAT\Result_Timestamped\GeoTIFF\TEST\Clip_Vorarlberg\2018-12-31 12_00_00_IR_VIS_WR.tif    Convert NAT-File to GeoTIFF: C:\

C:\Users\Andreas\anaconda3\envs\scikit-learn\lib\site-packages\pyproj\crs\crs.py:543: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj_string = self.to_proj4()
C:\Users\Andreas\anaconda3\envs\scikit-learn\lib\site-packages\pyproj\crs\crs.py:543: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj_string = self.to_proj4()
C:\Users\Andreas\anaconda3\envs\scikit-learn\lib\site-packages\pyproj\crs\crs.py:543: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj_string = se

       Band values: VIS006 0.7094101       Band values: 6.76026
 VIS006 0.66768 5.88393
       Band values:        Band values:VIS006 0.91806006 8.346001
 VIS006 0.79287004 7.3236156
       Execute Band VIS008 as Bandnr.2       Execute Band VIS008 as Bandnr.2       Execute Band VIS008 as Bandnr.2


       Execute Band VIS008 as Bandnr.2


...

       Band values: IR_134 46.33643 77.542595
    GeoTIFF-File C:\Users\Andreas\Documents\UNIGIS\2017\Master-Thesis\Daten\Satellite\EUMETSAT\Result_Timestamped\GeoTIFF\TEST\Clip_Lake_of_Constance\2018-12-31 16_15_00_HRV.tif not exists!
    Convert NAT-File to GeoTIFF: C:\Users\Andreas\Documents\UNIGIS\2017\Master-Thesis\Daten\Satellite\EUMETSAT\Result_Timestamped\GeoTIFF\TEST\Clip_Lake_of_Constance\2018-12-31 16_15_00_HRV.tif
       Execute Band HRV as Bandnr.1


C:\Users\Andreas\anaconda3\envs\scikit-learn\lib\site-packages\pyproj\crs\crs.py:543: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj_string = self.to_proj4()


       Band values: IR_134 48.22771 77.85781
    GeoTIFF-File C:\Users\Andreas\Documents\UNIGIS\2017\Master-Thesis\Daten\Satellite\EUMETSAT\Result_Timestamped\GeoTIFF\TEST\Clip_Lake_of_Constance\2018-12-31 16_00_00_HRV.tif not exists!
    Convert NAT-File to GeoTIFF: C:\Users\Andreas\Documents\UNIGIS\2017\Master-Thesis\Daten\Satellite\EUMETSAT\Result_Timestamped\GeoTIFF\TEST\Clip_Lake_of_Constance\2018-12-31 16_00_00_HRV.tif
       Execute Band HRV as Bandnr.1


C:\Users\Andreas\anaconda3\envs\scikit-learn\lib\site-packages\pyproj\crs\crs.py:543: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj_string = self.to_proj4()


       Band values: HRV 0.0 0.10571551
### NAT-File: C:\Users\Andreas\Documents\UNIGIS\2017\Master-Thesis\Daten\Satellite\EUMETSAT\Native\MSG4-SEVI-MSG15-0100-NA-20181231162742.767000000Z-20181231162800-1404462-6.nat
    Deleting NAT-File: C:\Users\Andreas\Documents\UNIGIS\2017\Master-Thesis\Daten\Satellite\EUMETSAT\Native\MSG4-SEVI-MSG15-0100-NA-20181231162742.767000000Z-20181231162800-1404462-6.nat
       Band values: HRV 0.0 0.14095402
### NAT-File: C:\Users\Andreas\Documents\UNIGIS\2017\Master-Thesis\Daten\Satellite\EUMETSAT\Native\MSG4-SEVI-MSG15-0100-NA-20181231161243.024000000Z-20181231161300-1404462-6.nat
    Deleting NAT-File: C:\Users\Andreas\Documents\UNIGIS\2017\Master-Thesis\Daten\Satellite\EUMETSAT\Native\MSG4-SEVI-MSG15-0100-NA-20181231161243.024000000Z-20181231161300-1404462-6.nat
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
